# RSV, COVID-19 and Influenza (2020 - 2026) 
    by Saira Begum, Samirah Begum and Maia-Eve Medina

### Dashboard Outline
This dashboard visualises hospital admissions and PCR positivity for Respiratory syncytial virus (RSV), Coronavirus disease 2019 (COVID-19) and Influenza. The metrics used for comparison are: hospital admissions rate and percentage of positive PCR tests (7 day period). 

In [8]:
from IPython.display import display, Markdown

sections = [
    "Bar Chart",
    "Heatmap",
    "Line Graph",
]

text = "### Contents\n\n"
text = text + "<details>\n<summary><b>Open contents</b></summary>\n\n"
for title in sections:
    anchor = title.replace(" ", "-")
    text = text + "- [" + title + "](#" + anchor + ")\n"
text = text + "\n</details>"

display(Markdown(text))

### Contents

<details>
<summary><b>Open contents</b></summary>

- [Bar Chart](#Bar-Chart)
- [Heatmap](#Heatmap)
- [Line Graph](#Line-Graph)

</details>

In [5]:
display(Markdown("## Bar Chart"))


## Bar Chart

In [10]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

# ---------- settings ----------
years_to_use = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
england_population = 57000000   # approximate, check the figure for England (ONS)

months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

files = {
    "positivity": {
        "COVID-19": "covid_positivity.json",
        "Influenza": "influenza_positivity.json",
        "RSV": "RSV_positivity.json",
    },
    "admissions": {
        "COVID-19": "covid_admissions.json",
        "Influenza": "influenza_admissions.json",
        "RSV": "RSV_cases.json",
    },
}

legend_labels = {
    "positivity": {
        "COVID-19": "COVID-19",
        "Influenza": "Influenza",
        "RSV": "RSV",
    },
    "admissions": {
        "COVID-19": "COVID-19 (converted from daily counts)",
        "Influenza": "Influenza",
        "RSV": "RSV",
    },
}

colours = {"COVID-19": "tab:pink", "Influenza": "tab:blue", "RSV": "tab:purple"}

y_labels = {
    "positivity": "Average PCR positivity (%)",
    "admissions": "Average weekly hospital admission rate per 100,000",
}
titles = {
    "positivity": "Average Monthly PCR positivity (2020 - 2026)",
    "admissions": "Average Monthly Hospital Admissions (2020 - 2026)",
}

# ---------- work out the averages ----------
def month_averages(filename):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for month in months:
        totals[month] = 0
        counts[month] = 0

    for record in records:
        if record["year"] in years_to_use:
            month = record["month"]
            totals[month] = totals[month] + record["metric_value"]
            counts[month] = counts[month] + 1

    averages = []
    for month in months:
        if counts[month] > 0:
            averages.append(totals[month] / counts[month])
        else:
            averages.append(0)
    return averages

data = {}
for metric in files:
    data[metric] = {}
    for disease in files[metric]:
        data[metric][disease] = month_averages(files[metric][disease])

# convert COVID-19 daily counts into a weekly rate per 100,000
# (keep this in the same cell as the loop above, or it converts twice)
for i in range(len(months)):
    daily = data["admissions"]["COVID-19"][i]
    data["admissions"]["COVID-19"][i] = daily * 7 / england_population * 100000

# ---------- draw the chart ----------
def draw_chart(metric):
    diseases = list(files[metric])
    width = 0.8 / len(diseases)

    plt.figure(figsize=(12, 5))
    n = 0
    for disease in diseases:
        positions = []
        for i in range(len(months)):
            positions.append(i + n * width)
        plt.bar(positions, data[metric][disease], width=width,
                label=legend_labels[metric][disease], color=colours[disease])
        n = n + 1

    centre = []
    for i in range(len(months)):
        centre.append(i + width * (len(diseases) - 1) / 2)
    plt.xticks(centre, month_names)

    plt.xlabel("Month")
    plt.ylabel(y_labels[metric])
    plt.title(titles[metric])
    plt.ylim(bottom=0)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.show()

# ---------- the toggle ----------
toggle = widgets.ToggleButtons(
    options=[("Positive tests", "positivity"), ("Hospital admissions", "admissions")],
    description="Chart:",
)

widgets.interact(draw_chart, metric=toggle)

interactive(children=(ToggleButtons(description='Chart:', options=(('Positive tests', 'positivity'), ('Hospita…

<function __main__.draw_chart(metric)>

Bar charts show:

average hospital admissions for each month from 2020 - 2026, for COVID-19, Influenza and RSV.
average PCR positive tests for each month from 2020 - 2026, for COVID-19, Influenza and RSV. Trend shows all are more prevalent in winter but COVID-19 follows trend less strictly.

Toggle between Positive Tests chart and Hospital Admissions chart.

In [14]:
display(Markdown("[Back to contents](#Contents)"))

[Back to contents](#Contents)

In [11]:
display(Markdown("## Heatmap"))

## Heatmap

In [11]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

files = {
    "COVID-19": "covid_positivity.json",
    "Influenza": "influenza_positivity.json",
    "RSV": "RSV_positivity.json",
}

# every month of every year, in order
timeline = []
for year in years:
    for month in months:
        timeline.append((year, month))

# one average for every month on the timeline
def monthly_row(filename):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for key in timeline:
        totals[key] = 0
        counts[key] = 0

    for record in records:
        key = (record["year"], record["month"])
        if key in totals:
            totals[key] = totals[key] + record["metric_value"]
            counts[key] = counts[key] + 1

    row = []
    for key in timeline:
        if counts[key] > 0:
            row.append(totals[key] / counts[key])
        else:
            row.append(float("nan"))
    return row

# work out each disease's row once
all_rows = {}
for disease in files:
    all_rows[disease] = monthly_row(files[disease])

# label each January with its year
tick_positions = []
tick_labels = []
for i in range(len(timeline)):
    if timeline[i][1] == 1:
        tick_positions.append(i)
        tick_labels.append(timeline[i][0])

def draw_heatmap(selected):
    if len(selected) == 0:
        print("Select at least one disease")
        return

    grid = []
    for disease in selected:
        grid.append(all_rows[disease])

    plt.figure(figsize=(14, 1.5 + 1.2 * len(selected)))
    image = plt.imshow(grid, cmap="YlOrRd", aspect="auto")

    plt.xticks(tick_positions, tick_labels)
    plt.yticks(range(len(selected)), selected)
    plt.xlabel("Year (each column is one month)")
    plt.title("PCR Positivity by Month (2020 - 2026)")
    plt.colorbar(image, label="Average PCR positivity (%)")
    plt.show()

selector = widgets.SelectMultiple(
    options=list(files),
    value=list(files),
    description="Diseases",
)

widgets.interact(draw_heatmap, selected=selector)

interactive(children=(SelectMultiple(description='Diseases', index=(0, 1, 2), options=('COVID-19', 'Influenza'…

<function __main__.draw_heatmap(selected)>

Heatmap shows average PCR positive tests every month over years 2020 - 2026. 

Peaks in winter, COVID more prevalent from 2020 and follows trend less strictly. 

In [15]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
england_population = 57000000   # approximate, check the figure for England (ONS)

files = {
    "COVID-19": "covid_admissions.json",
    "Influenza": "influenza_admissions.json",
    "RSV": "RSV_cases.json",
}

# multiply each value by this to get a weekly rate per 100,000
scale = {
    "COVID-19": 7 / england_population * 100000,   # daily count -> weekly rate
    "Influenza": 1,
    "RSV": 1,
}

# every month of every year, in order
timeline = []
for year in years:
    for month in months:
        timeline.append((year, month))

# one average for every month on the timeline
def monthly_row(filename, factor):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for key in timeline:
        totals[key] = 0
        counts[key] = 0

    for record in records:
        key = (record["year"], record["month"])
        if key in totals:
            totals[key] = totals[key] + record["metric_value"] * factor
            counts[key] = counts[key] + 1

    row = []
    for key in timeline:
        if counts[key] > 0:
            row.append(totals[key] / counts[key])
        else:
            row.append(float("nan"))
    return row

# work out each disease's row once
all_rows = {}
for disease in files:
    all_rows[disease] = monthly_row(files[disease], scale[disease])

# label each January with its year
tick_positions = []
tick_labels = []
for i in range(len(timeline)):
    if timeline[i][1] == 1:
        tick_positions.append(i)
        tick_labels.append(timeline[i][0])

def draw_heatmap(selected):
    if len(selected) == 0:
        print("Select at least one disease")
        return

    grid = []
    for disease in selected:
        grid.append(all_rows[disease])

    plt.figure(figsize=(14, 1.5 + 1.2 * len(selected)))
    image = plt.imshow(grid, cmap="YlOrRd", aspect="auto")

    plt.xticks(tick_positions, tick_labels)
    plt.yticks(range(len(selected)), selected)
    plt.xlabel("Year  (month per column)")
    plt.title("Hospital Admissions by Month (2020 - 2026)")
    plt.colorbar(image, label="Average weekly admission rate per 100,000")
    plt.show()

selector = widgets.SelectMultiple(
    options=list(files),
    value=list(files),
    description="Diseases",
)

widgets.interact(draw_heatmap, selected=selector)

interactive(children=(SelectMultiple(description='Diseases', index=(0, 1, 2), options=('COVID-19', 'Influenza'…

<function __main__.draw_heatmap(selected)>

Heat map shows hospital admissions each month from 2020 - 2026 for COVID-19, Influenza and RSV. 

COVID-19 hospital admissions greater in 2020.
Heatmap tends to be warmer towards winter times (seasonal).  

Select/deselect diseases to view above. 

In [12]:
display(Markdown("[Back to contents](#Contents)"))

[Back to contents](#Contents)

In [14]:
display(Markdown("## Line Graph"))

## Line Graph

Use the toggle to switch between the admissions rates and pcr positivity. Each point on the graph represents a month and each line represents the virus. To view the trend of a specific virus, choose which to unselect. Data obtained is from the UKHSA data dashboard. 

Influenza and RSV follow a clear winter pattern, with peaks around November to January. Covid-19 has been less seasonal, with waves at varying times of year.

In [13]:
import json
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
england_population = 57000000   # approximate, check the figure for England (ONS)

colours = {"COVID-19": "tab:pink", "Influenza": "tab:blue", "RSV": "tab:purple"}

# everything that differs between the two graphs lives here
measures = {
    "Admissions": {
        "files": {
            "COVID-19": "covid_admissions.json",
            "Influenza": "influenza_admissions.json",
            "RSV": "RSV_cases.json",
        },
        # multiply each value by this to get a weekly rate per 100,000
        "scale": {
            "COVID-19": 7 / england_population * 100000,   # daily count -> weekly rate
            "Influenza": 1,
            "RSV": 1,
        },
        "ylabel": "Average weekly admission rate per 100,000",
        "title": "Monthly hospital admissions, 2020-2026",
    },
    "PCR positivity": {
        # edit these names if your files are called something different
        "files": {
            "COVID-19": "covid_positivity.json",
            "Influenza": "influenza_positivity.json",
            "RSV": "RSV_positivity.json",
        },
        "scale": {"COVID-19": 1, "Influenza": 1, "RSV": 1},   # already a percentage
        "ylabel": "Average PCR positivity (%)",
        "title": "Monthly PCR positivity, 2020-2026",
    },
}

def get_year(record):
    if "year" in record:
        return int(record["year"])
    return int(record["date"][0:4])      # "2023-03-14" -> 2023

def get_month(record):
    if "month" in record:
        return int(record["month"])
    return int(record["date"][5:7])      # "2023-03-14" -> 3

# list of (first day of month, average value) for every month in 2020-2026
def monthly_series(filename, factor):
    with open(filename) as f:
        records = json.load(f)

    dates = []
    values = []
    for year in years:
        for month in range(1, 13):
            total = 0
            count = 0
            for record in records:
                if get_year(record) == year and get_month(record) == month:
                    total = total + record["metric_value"] * factor
                    count = count + 1
            dates.append(datetime.date(year, month, 1))
            if count > 0:
                values.append(total / count)
            else:
                values.append(float("nan"))    # no data -> gap in the line
    return dates, values

# work out every series once: all_series[measure][disease]
all_series = {}
for measure in measures:
    all_series[measure] = {}
    for disease in measures[measure]["files"]:
        filename = measures[measure]["files"][disease]
        if not os.path.exists(filename):
            print("Missing file, skipping:", filename)
            continue
        factor = measures[measure]["scale"][disease]
        all_series[measure][disease] = monthly_series(filename, factor)

# draw the chosen measure for the selected diseases
def draw_chart(measure, selected):
    available = [d for d in selected if d in all_series[measure]]
    if len(available) == 0:
        print("Nothing to show: select a disease that has a data file")
        return

    fig, ax = plt.subplots(figsize=(14, 5))
    for disease in available:
        dates, values = all_series[measure][disease]
        ax.plot(dates, values, marker="o", markersize=3,
                color=colours[disease], label=disease)

    # one label per year, with a light tick for every month
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())

    ax.set_xlabel("Year (each point represents a month)")
    ax.set_ylabel(measures[measure]["ylabel"])
    ax.set_title(measures[measure]["title"])
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

measure_toggle = widgets.ToggleButtons(
    options=list(measures),
    value="Admissions",
    description="Show:",
)

# one checkbox per virus, all ticked to start
checkboxes = {}
for disease in colours:
    checkboxes[disease] = widgets.Checkbox(value=True, description=disease)

show_all_button = widgets.Button(description="Show all")

def show_all(button):
    for disease in checkboxes:
        checkboxes[disease].value = True

show_all_button.on_click(show_all)

# collect the ticked viruses and redraw whenever anything changes
def update(measure, **ticks):
    selected = [d for d in ticks if ticks[d]]
    draw_chart(measure, selected)

controls = {"measure": measure_toggle}
for disease in checkboxes:
    controls[disease] = checkboxes[disease]

out = widgets.interactive_output(update, controls)

display(widgets.VBox([
    measure_toggle,
    widgets.HBox(list(checkboxes.values()) + [show_all_button]),
    out,
]))

In [15]:
display(Markdown("[Back to contents](#Contents)"))

[Back to contents](#Contents)